# 🧠 EEG Preprocessing Pipeline — Stage 4: Epoching

**Dataset:** `ds006780` (ASD EEG, BIDS-formatted)
**Input (read from GCS or local):**
- Stage 3: `desc-clean_eeg.fif` — average-referenced, ICA-cleaned **continuous** EEG

**Output (written to GCS or local):**
- `desc-epo.fif` — MNE `Epochs` (segmented; no rejection, no baseline correction)

---

## What this stage does

```
 Stage 3 clean EEG  (continuous, avg-ref, ICA-cleaned)
        │
        ▼
 per-task EPOCH_CONFIG
   • Restingstate         → fixed-length windows (make_fixed_length_epochs)
   • FAST / IC / motor    → event-related epochs from MNE annotations
        │                    (events_from_annotations, configured labels only)
        ▼
 mne.Epochs   baseline=None, reject=None   (keep all; QC stats logged)
        │
        ▼
 desc-epo.fif
```

**Events come only from the annotations carried in the Stage 3 `.fif`** — originally the
BIDS `events.tsv`, loaded by `read_raw_bids` in Stage 1 and preserved through filtering,
resampling, and ICA. There is **no** separate BIDS/S3 event path: a recording with no usable
annotations fails loudly so the upstream pipeline can be investigated.

This stage is **epoching only** — no PSD / band-power / connectivity / feature extraction.


# 1. Setup

## 1.1 Install dependencies

In [ ]:
#!pip install mne pandas numpy joblib tqdm \
#            google-cloud-storage wandb

## 1.2 Imports + sanity check

In [2]:
import gc
import json
import time
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import nullcontext

import numpy as np
import pandas as pd
import mne
from joblib import Parallel, delayed
from tqdm.auto import tqdm

try:
    from google.cloud import storage as gcs_lib
    HAS_GCS = True
except ImportError:
    HAS_GCS = False

mne.set_log_level('WARNING')
print(f'MNE version:    {mne.__version__}')
print(f'GCS available:  {HAS_GCS}')

MNE version:    1.10.2
GCS available:  True


# 2. Configuration

## 2.1 Paths

In [3]:
PROJECT_ROOT     = Path.home() / 'asd_eeg_pipeline'
DATASET_ID       = 'ds006780'

STAGE3_PIPELINE  = 'mne-ica-apply'        # input: cleaned EEG
PIPELINE_NAME    = 'mne-epochs'           # this stage's output

STAGE3_LOCAL     = PROJECT_ROOT / 'derivatives' / STAGE3_PIPELINE
DERIV_ROOT       = PROJECT_ROOT / 'derivatives' / PIPELINE_NAME
LOG_DIR          = PROJECT_ROOT / 'derivatives' / 'logs'

for d in (DERIV_ROOT, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'📥 Input  (Stage 3):  {STAGE3_LOCAL}')
print(f'📤 Output (Stage 4):  {DERIV_ROOT}')

📥 Input  (Stage 3):  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-apply
📤 Output (Stage 4):  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-epochs


## 2.2 GCS configuration

In [4]:
from asd_eeg import gcs_io

INPUT_SOURCE  = 'gcs'      # 'gcs' | 'local'
UPLOAD_TO_GCS = True

# GCS I/O is centralized in gcs_io.GCSStore (bucket / dataset / retry configured once).
store   = gcs_io.GCSStore()
HAS_GCS = gcs_io.HAS_GCS

print(f'INPUT_SOURCE   = {INPUT_SOURCE}')
print(f'UPLOAD_TO_GCS  = {UPLOAD_TO_GCS}')
print(f'GCS Stage 3    = gs://{store.bucket_name}/{store.prefix(STAGE3_PIPELINE)}/')
print(f'GCS Stage 4    = gs://{store.bucket_name}/{store.prefix(PIPELINE_NAME)}/')

INPUT_SOURCE   = gcs
UPLOAD_TO_GCS  = True
GCS Stage 3    = gs://asd-eeg-dataset/derivatives/ds006780/mne-ica-apply/
GCS Stage 4    = gs://asd-eeg-dataset/derivatives/ds006780/mne-epochs/


In [5]:
if INPUT_SOURCE == 'gcs' or UPLOAD_TO_GCS:
    if not HAS_GCS:
        raise RuntimeError(
            'google-cloud-storage is not installed. '
            'Run: pip install google-cloud-storage')
    print(f'✅ GCS store ready  →  gs://{store.bucket_name}/  (available: {store.available()})')
else:
    print('ℹ️  GCS not needed (INPUT_SOURCE=local, UPLOAD_TO_GCS=False)')

✅ GCS store ready  →  gs://asd-eeg-dataset/  (available: True)


## 2.3 Epoching policy

Per-task segmentation, the Stage-4 analogue of Stage 1's `TASK_CONFIG`. Each event task
names **which annotation labels to epoch on** (`events`), so non-stimulus markers
(`Response_button`, `Pause`, `fixation_cross`, …) are excluded by default.

> ⚠️ **`tmin` / `tmax` are PROVISIONAL** — validate them against the actual task event
> timing before any production run. Baseline correction and epoch rejection are left
> **off** on purpose (downstream-analysis decisions); QC amplitude stats are logged instead.

In [6]:
GLOBAL_BASELINE = None   # no baseline correction — downstream analysis decides
GLOBAL_REJECT   = None   # no rejection — keep all epochs, log amplitude stats only

EPOCH_CONFIG = {
    'Restingstate': dict(mode='fixed', duration=2.0, overlap=0.0),
    # tmin/tmax PROVISIONAL until validated against task event timing.
    'FAST':  dict(mode='event', tmin=-0.2, tmax=0.8,
                  events=['Face_upright', 'Face_inverted',
                          'Face_upright_shadow', 'Face_inverted_shadow',
                          'Object_upright', 'Object_inverted',
                          'Object_upright_shadow', 'Object_inverted_shadow']),
    'IC':    dict(mode='event', tmin=-0.2, tmax=0.8,
                  events=['Illusory_contours', 'Non_Illusory_contours']),
    'motor': dict(mode='event', tmin=-0.2, tmax=0.8, events=['button_press']),
}

print('Epoching policy (per task):')
for t, c in EPOCH_CONFIG.items():
    if c['mode'] == 'fixed':
        print(f'  {t:14s} fixed  duration={c["duration"]}s overlap={c.get("overlap", 0.0)}s')
    else:
        print(f'  {t:14s} event  tmin={c["tmin"]} tmax={c["tmax"]}  labels={c["events"]}')
print(f'\nBaseline: {GLOBAL_BASELINE}   Reject: {GLOBAL_REJECT}   '
      f'(keep all epochs; amplitude stats logged for QC)')

Epoching policy (per task):
  Restingstate   fixed  duration=2.0s overlap=0.0s
  FAST           event  tmin=-0.2 tmax=0.8  labels=['Face_upright', 'Face_inverted', 'Face_upright_shadow', 'Face_inverted_shadow', 'Object_upright', 'Object_inverted', 'Object_upright_shadow', 'Object_inverted_shadow']
  IC             event  tmin=-0.2 tmax=0.8  labels=['Illusory_contours', 'Non_Illusory_contours']
  motor          event  tmin=-0.2 tmax=0.8  labels=['button_press']

Baseline: None   Reject: None   (keep all epochs; amplitude stats logged for QC)


## 2.4 Tasks and execution

In [7]:
TASKS        = ['Restingstate', 'FAST', 'IC', 'motor']   # or None for all
N_JOBS_OUTER = 1                                          # parallel files
OVERWRITE    = False                                      # redo finished files?

print(f'TASKS        = {TASKS}')
print(f'N_JOBS_OUTER = {N_JOBS_OUTER}')
print(f'OVERWRITE    = {OVERWRITE}')

TASKS        = ['Restingstate', 'FAST', 'IC', 'motor']
N_JOBS_OUTER = 1
OVERWRITE    = False


## 2.5 Experiment tracking (Weights & Biases)

In [8]:
from asd_eeg import wandb_tracking as wbt

ENABLE_WANDB = True          # master switch — False makes all tracking a no-op
WANDB_MODE   = 'online'      # 'online' | 'offline' | 'disabled'
WANDB_GROUP  = None          # use the SAME id as Stages 1-3 to group the pipeline

print(f'W&B tracking: {ENABLE_WANDB}  (mode={WANDB_MODE}, available={wbt.HAS_WANDB})')

W&B tracking: True  (mode=online, available=True)


# 3. Initialize BIDS-derivatives dataset

In [9]:
def init_bids_derivatives(deriv_root, pipeline_name):
    desc = {
        'Name': f'{DATASET_ID} — {pipeline_name}',
        'BIDSVersion': '1.9.0',
        'DatasetType': 'derivative',
        'GeneratedBy': [{
            'Name': pipeline_name,
            'Version': '0.1.0',
            'Description': (
                'Epoching of ICA-cleaned EEG. Restingstate -> fixed-length epochs; '
                'event tasks -> event-related epochs from annotations (configured '
                'labels only). No baseline correction, no rejection.'
            ),
            'CodeURL': 'local notebook',
        }],
        'SourceDatasets': [{'URL': f'derivatives/{STAGE3_PIPELINE}'}],
    }
    out = Path(deriv_root) / 'dataset_description.json'
    out.write_text(json.dumps(desc, indent=2))
    return out

desc_path = init_bids_derivatives(DERIV_ROOT, PIPELINE_NAME)
print(f'✅ Wrote {desc_path}')

✅ Wrote /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-epochs/dataset_description.json


# 4. Core pipeline functions

## 4.1 BIDS filename parser

In [10]:
# Reuse the shared parsers (same ones Stages 2 & 3 use) instead of re-defining.
parse_bids_filename = gcs_io.parse_bids_filename
make_key            = gcs_io.make_key

demo = parse_bids_filename('sub-10003_task-FAST_run-01_desc-clean_eeg.fif')
print(demo)

{'subject': '10003', 'task': 'FAST', 'run': '01', 'desc': 'clean'}


## 4.2 Discovery — Stage 3 clean files

In [11]:
def discover_local(stage3_root, tasks=None):
    """Find (clean_path, entities) on local disk."""
    files = sorted(Path(stage3_root).rglob('*_desc-clean_eeg.fif'))
    out = []
    for f in files:
        ent = parse_bids_filename(f.name)
        if tasks is not None and ent['task'] not in tasks:
            continue
        out.append((f, ent))
    return out


def discover_gcs(stage3_pipeline=STAGE3_PIPELINE, tasks=None):
    """Find (clean_blob, entities) in GCS via the shared store."""
    return store.list_files(stage3_pipeline, '_desc-clean_eeg.fif', tasks=tasks)

## 4.3 GCS staging helpers

In [12]:
# Thin wrappers over the shared gcs_io.GCSStore so process_one stays unchanged.
staged_from_gcs = store.staged          # download → yield → cleanup contextmanager


def upload_to_gcs(local_path):
    """Upload one Stage 4 output to GCS. Skips if size matches existing blob."""
    return store.upload(local_path, DERIV_ROOT, PIPELINE_NAME)

## 4.4 Loading

In [13]:
def load_clean(fpath):
    return mne.io.read_raw_fif(fpath, preload=True, verbose=False)

## 4.5 Build epochs

In [14]:
def select_event_id(raw, wanted):
    """Map configured annotation labels -> integer codes, keeping only those present.

    Returns (events, event_id). Raises if the recording carries no annotations, or none
    of the configured labels are present. Stage 4 depends solely on the Stage 3
    annotations — there is intentionally no BIDS/S3 fallback.
    """
    if len(raw.annotations) == 0:
        raise ValueError('no annotations on this recording (events_from_annotations empty)')
    events, full_id = mne.events_from_annotations(raw, verbose=False)
    event_id = {str(lbl): code for lbl, code in full_id.items() if lbl in wanted}
    if not event_id:
        raise ValueError(
            f'none of the configured event labels {sorted(wanted)} are present; '
            f'available: {sorted(map(str, full_id))}')
    return events, event_id


def build_epochs(raw, task):
    """Segment per EPOCH_CONFIG[task]. Returns (epochs, mode, event_types)."""
    cfg = EPOCH_CONFIG.get(task)
    if cfg is None:
        raise ValueError(f'no EPOCH_CONFIG entry for task {task!r}')
    mode = cfg['mode']

    if mode == 'fixed':
        epochs = mne.make_fixed_length_epochs(
            raw, duration=cfg['duration'], overlap=cfg.get('overlap', 0.0),
            preload=True, verbose=False)
        return epochs, mode, []

    if mode == 'event':
        events, event_id = select_event_id(raw, set(cfg['events']))
        epochs = mne.Epochs(
            raw, events, event_id=event_id,
            tmin=cfg['tmin'], tmax=cfg['tmax'],
            baseline=GLOBAL_BASELINE, reject=GLOBAL_REJECT,
            preload=True, verbose=False)
        return epochs, mode, sorted(event_id)

    raise ValueError(f'unknown epoch mode {mode!r} for task {task!r}')

## 4.6 Output paths and savers

In [15]:
def stage4_output_dir(entities, deriv_root=DERIV_ROOT):
    return Path(deriv_root) / f'sub-{entities["subject"]}' / 'eeg'


def stage4_basename(entities):
    parts = [f'sub-{entities["subject"]}', f'task-{entities["task"]}']
    if entities.get('run'):
        parts.append(f'run-{entities["run"]}')
    return '_'.join(parts)


def epochs_path(entities, deriv_root=DERIV_ROOT):
    out_dir = stage4_output_dir(entities, deriv_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    # '..._desc-epo.fif' ends with '-epo.fif', satisfying MNE's epochs-filename check.
    return out_dir / f'{stage4_basename(entities)}_desc-epo.fif'


def save_epochs(epochs, entities, deriv_root=DERIV_ROOT):
    out = epochs_path(entities, deriv_root)
    epochs.save(out, overwrite=True, verbose=False)
    return out

## 4.7 QC stats (amplitude only — no rejection)

In [16]:
def epoch_qc_stats(epochs):
    """Per-epoch peak-to-peak amplitude summary in µV (QC only — nothing is dropped)."""
    data = epochs.get_data(copy=False)            # (n_epochs, n_channels, n_times), volts
    if data.size == 0:
        return {'mean_p2p_uv': 0.0, 'max_p2p_uv': 0.0}
    ptp = (data.max(axis=-1) - data.min(axis=-1)) * 1e6     # µV, per epoch×channel
    return {
        'mean_p2p_uv': round(float(ptp.mean()), 2),
        'max_p2p_uv':  round(float(ptp.max()), 2),
    }

## 4.8 Logging — fixed-schema CSV

In [17]:
LOG_FILE  = LOG_DIR / '04_epoching_log.csv'
ERROR_LOG = LOG_DIR / '04_epoching_errors.txt'

# Must match asd_eeg.utils.logs.STAGE4_LOG_COLUMNS exactly.
LOG_COLUMNS = [
    'timestamp', 'subject', 'task', 'run', 'status',
    'epoch_mode', 'n_epochs', 'epoch_duration_s', 'tmin', 'tmax',
    'n_event_types', 'event_types', 'sfreq', 'n_channels',
    'mean_p2p_uv', 'max_p2p_uv', 'total_duration_s',
    'epochs_path', 'epochs_gcs', 'warning', 'error',
]


def append_log(record, log_path=LOG_FILE):
    """Append one row, padded to LOG_COLUMNS."""
    record = {**record, 'timestamp': datetime.now().isoformat(timespec='seconds')}
    row = {col: record.get(col, '') for col in LOG_COLUMNS}
    df = pd.DataFrame([row], columns=LOG_COLUMNS)
    header = not Path(log_path).exists()
    df.to_csv(log_path, mode='a', header=header, index=False)

print(f'Log file:   {LOG_FILE}')
print(f'Error log:  {ERROR_LOG}')
print(f'Columns:    {len(LOG_COLUMNS)}')

Log file:   /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/04_epoching_log.csv
Error log:  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/04_epoching_errors.txt
Columns:    21


## 4.9 The orchestrator — `process_one`

In [18]:
def process_one(item, deriv_root=DERIV_ROOT, log_file=LOG_FILE):
    """
    item: (clean_ref, entities)
      clean_ref is a local Path or a GCS blob name depending on INPUT_SOURCE.
    """
    clean_ref, entities = item
    base = {k: entities.get(k) for k in ('subject', 'task', 'run')}
    out_epo = epochs_path(entities, deriv_root)

    if not OVERWRITE and out_epo.exists():
        result = {**base, 'status': 'skipped', 'epochs_path': str(out_epo)}
        if log_file is not None:
            append_log(result, log_file)
        return result

    raw = epochs = None
    try:
        clean_cm = (staged_from_gcs(clean_ref) if INPUT_SOURCE == 'gcs'
                    else nullcontext(clean_ref))
        with clean_cm as clean_path:
            raw = load_clean(clean_path)
            epochs, mode, event_types = build_epochs(raw, entities['task'])
            save_epochs(epochs, entities, deriv_root)

        epochs_gcs = upload_to_gcs(out_epo) if UPLOAD_TO_GCS else ''

        qc = epoch_qc_stats(epochs)
        n_epochs = len(epochs)
        epoch_len = float(epochs.times[-1] - epochs.times[0])
        result = {
            **base,
            'status':           'ok',
            'epoch_mode':       mode,
            'n_epochs':         n_epochs,
            'epoch_duration_s': round(epoch_len, 3),
            'tmin':             round(float(epochs.tmin), 3),
            'tmax':             round(float(epochs.tmax), 3),
            'n_event_types':    len(event_types),
            'event_types':      ','.join(map(str, event_types)),
            'sfreq':            float(raw.info['sfreq']),
            'n_channels':       len(epochs.ch_names),
            'mean_p2p_uv':      qc['mean_p2p_uv'],
            'max_p2p_uv':       qc['max_p2p_uv'],
            'total_duration_s': round(n_epochs * epoch_len, 1),
            'epochs_path':      str(out_epo),
            'epochs_gcs':       epochs_gcs,
            'warning':          '' if n_epochs > 0 else 'no_epochs',
        }

    except Exception as e:
        result = {**base, 'status': 'fail', 'error': str(e)}
        with open(ERROR_LOG, 'a') as f:
            f.write(f'\n=== {entities} ===\n{traceback.format_exc()}\n')
    finally:
        if raw is not None:
            del raw
        if epochs is not None:
            del epochs
        gc.collect()

    if log_file is not None:
        append_log(result, log_file)
    return result

# 5. Find input files

In [19]:
if INPUT_SOURCE == 'gcs':
    inputs = discover_gcs(STAGE3_PIPELINE, tasks=TASKS)
    print(f'Found {len(inputs)} clean files in GCS')
else:
    inputs = discover_local(STAGE3_LOCAL, tasks=TASKS)
    print(f'Found {len(inputs)} clean files locally')

# Stage 4 only epochs tasks it has a config for; anything else fails loudly per file.
unconfigured = sorted({ent['task'] for _, ent in inputs} - set(EPOCH_CONFIG))
if unconfigured:
    print(f'\n⚠️  No EPOCH_CONFIG for tasks {unconfigured} — those files will fail (status=fail).')

subjects   = {ent['subject'] for _, ent in inputs}
tasks_seen = {ent['task']    for _, ent in inputs}
print(f'\n  Subjects: {len(subjects)}')
print(f'  Tasks:    {sorted(tasks_seen)}')
for ref, _ in inputs[:3]:
    name = ref.name if hasattr(ref, 'name') else Path(ref).name
    print(f'  {name}')

Found 2156 clean files in GCS

  Subjects: 136
  Tasks:    ['FAST', 'IC', 'Restingstate', 'motor']
  sub-10003_task-FAST_run-01_desc-clean_eeg.fif
  sub-10003_task-IC_run-01_desc-clean_eeg.fif
  sub-10003_task-Restingstate_run-01_desc-clean_eeg.fif


# 6. Smoke test on one file per mode

Run one **fixed** (Restingstate) and one **event** (FAST/IC/motor) recording, then reload
the saved epochs to confirm both modes work end-to-end before the full batch.

In [20]:
# One representative file per available task.
by_task = {}
for it in inputs:
    by_task.setdefault(it[1]['task'], it)

print('Tasks available for smoke test:', sorted(by_task))

# A fixed-length task + the first event task present.
smoke_tasks = []
if 'Restingstate' in by_task:
    smoke_tasks.append('Restingstate')
for t in ['FAST', 'IC', 'motor']:
    if t in by_task:
        smoke_tasks.append(t)
        break
print('Smoke-testing:', smoke_tasks)

Tasks available for smoke test: ['FAST', 'IC', 'Restingstate', 'motor']
Smoke-testing: ['Restingstate', 'FAST']


In [21]:
for t in smoke_tasks:
    item = by_task[t]
    ref = item[0]
    name = ref.name if hasattr(ref, 'name') else Path(ref).name
    print(f'\n🧪 [{t}] {name}')
    res = process_one(item, log_file=None)
    print(json.dumps(res, indent=2, default=str))


🧪 [Restingstate] sub-10003_task-Restingstate_run-01_desc-clean_eeg.fif
{
  "subject": "10003",
  "task": "Restingstate",
  "run": "01",
  "status": "ok",
  "epoch_mode": "fixed",
  "n_epochs": 31,
  "epoch_duration_s": 1.996,
  "tmin": 0.0,
  "tmax": 1.996,
  "n_event_types": 0,
  "event_types": "",
  "sfreq": 250.0,
  "n_channels": 68,
  "mean_p2p_uv": 38.07,
  "max_p2p_uv": 239.24,
  "total_duration_s": 61.9,
  "epochs_path": "/Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-epochs/sub-10003/eeg/sub-10003_task-Restingstate_run-01_desc-epo.fif",
  "epochs_gcs": "gs://asd-eeg-dataset/derivatives/ds006780/mne-epochs/sub-10003/eeg/sub-10003_task-Restingstate_run-01_desc-epo.fif",
  "warning": ""
}

🧪 [FAST] sub-10003_task-FAST_run-01_desc-clean_eeg.fif
{
  "subject": "10003",
  "task": "FAST",
  "run": "01",
  "status": "ok",
  "epoch_mode": "event",
  "n_epochs": 719,
  "epoch_duration_s": 1.0,
  "tmin": -0.2,
  "tmax": 0.8,
  "n_event_types": 8,
  "event_types": "Face_inverted,Fa

In [22]:
for t in smoke_tasks:
    out = epochs_path(by_task[t][1])
    if out.exists():
        epo = mne.read_epochs(out, preload=False, verbose=False)
        print(f'✅ [{t}] {out.name}')
        print(f'   n_epochs:   {len(epo)}')
        print(f'   times:      {epo.tmin:.2f} … {epo.tmax:.2f} s  ({len(epo.times)} samples)')
        print(f'   channels:   {len(epo.ch_names)}   sfreq: {epo.info["sfreq"]} Hz')
        print(f'   event_id:   {epo.event_id}')
        print(f'   file size:  {out.stat().st_size / 1e6:.1f} MB\n')
    else:
        print(f'❌ [{t}] no epochs file — check {ERROR_LOG}\n')

✅ [Restingstate] sub-10003_task-Restingstate_run-01_desc-epo.fif
   n_epochs:   31
   times:      0.00 … 2.00 s  (500 samples)
   channels:   68   sfreq: 250.0 Hz
   event_id:   {'1': 1}
   file size:  4.2 MB

✅ [FAST] sub-10003_task-FAST_run-01_desc-epo.fif
   n_epochs:   719
   times:      -0.20 … 0.80 s  (251 samples)
   channels:   68   sfreq: 250.0 Hz
   event_id:   {'Face_inverted': 1, 'Face_inverted_shadow': 2, 'Face_upright': 3, 'Face_upright_shadow': 4, 'Object_inverted': 5, 'Object_inverted_shadow': 6, 'Object_upright': 7, 'Object_upright_shadow': 8}
   file size:  49.1 MB



# 7. Run the full batch

In [23]:
import time as _time
import matplotlib.pyplot as plt
from collections import defaultdict

print(f'🚀 Stage 4 batch on {len(inputs)} files')
print(f'   Source:        {INPUT_SOURCE}')
print(f'   Upload to GCS: {UPLOAD_TO_GCS}')
print(f'   Tasks:         {sorted(EPOCH_CONFIG)}')
print(f'   N_JOBS_OUTER:  {N_JOBS_OUTER}')
print(f'   Output:        {DERIV_ROOT}')
print(f'   Log:           {LOG_FILE}\n')

# Flatten EPOCH_CONFIG for the W&B run config (one scalar per task/param).
epoch_cfg_flat = {f'epoch.{t}.{k}': v
                  for t, c in EPOCH_CONFIG.items() for k, v in c.items()}

run = wbt.init_run(
    stage=4,
    config={
        'dataset_id':    DATASET_ID,
        'pipeline_name': PIPELINE_NAME,
        'tasks':         TASKS,
        'input_source':  INPUT_SOURCE,
        'baseline':      str(GLOBAL_BASELINE),
        'reject':        str(GLOBAL_REJECT),
        'n_jobs_outer':  N_JOBS_OUTER,
        'overwrite':     OVERWRITE,
        **epoch_cfg_flat,
    },
    group=WANDB_GROUP,
    enabled=ENABLE_WANDB,
    mode=WANDB_MODE,
)

_t0 = _time.time()
if N_JOBS_OUTER == 1:
    results = [process_one(item) for item in tqdm(inputs, desc='epoching')]
else:
    results = Parallel(n_jobs=N_JOBS_OUTER, verbose=10)(
        delayed(process_one)(item) for item in inputs
    )

print('\n✅ Batch complete.')

# ── W&B: epoch counts + amplitude summary + artifacts ────────────────────────
ok_results = [r for r in results if r.get('status') == 'ok']

def _mean(key):
    vals = [r.get(key) for r in ok_results if isinstance(r.get(key), (int, float))]
    return round(sum(vals) / len(vals), 3) if vals else None

per_task = defaultdict(int)
for r in ok_results:
    if isinstance(r.get('n_epochs'), (int, float)):
        per_task[r['task']] += int(r['n_epochs'])

task_fig = None
if per_task:
    task_fig, _ax = plt.subplots(figsize=(7, 4))
    _ax.bar(list(per_task.keys()), list(per_task.values()))
    _ax.set_ylabel('total epochs')
    _ax.set_title('Stage 4 — epochs per task (dataset)')
    plt.setp(_ax.get_xticklabels(), rotation=30, ha='right')

wbt.finalize_run(
    run, results,
    namespace='quality',
    wall_seconds=_time.time() - _t0,
    log_csv=LOG_FILE,
    plots={'epochs_per_task': task_fig},
    quality={
        'n_epochs_mean':    _mean('n_epochs'),
        'n_epochs_total':   int(sum(per_task.values())),
        'mean_p2p_uv_mean': _mean('mean_p2p_uv'),
        **{f'epochs_total_{t}': v for t, v in per_task.items()},
    },
    histogram_columns=['n_epochs', 'mean_p2p_uv', 'total_duration_s'],
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/alirezafatemi/.netrc.


🚀 Stage 4 batch on 2156 files
   Source:        gcs
   Upload to GCS: True
   Tasks:         ['FAST', 'IC', 'Restingstate', 'motor']
   N_JOBS_OUTER:  1
   Output:        /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-epochs
   Log:           /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/04_epoching_log.csv



wandb: Currently logged in as: alirezafatemi98 (alirezafatemi98-messina-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoching:   0%|          | 0/2156 [00:00<?, ?it/s]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x11e23b380>> (for post_run_cell), with arguments args (<ExecutionResult object at 11e2d56a0, execution_count=23 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 118054520, raw_cell="import time as _time
import matplotlib.pyplot as p.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/alirezafatemi/Code/ASD_EEG_PROJECT/notebooks/04_epoching.ipynb#X64sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

# 8. Inspect the run

In [ ]:
from asd_eeg.utils import read_stage_log
log_df = read_stage_log(LOG_FILE)
print('Status counts:')
print(log_df['status'].value_counts(), '\n')

fails = log_df[log_df['status'] == 'fail']
if len(fails):
    print(f'⚠️  {len(fails)} failures — see {ERROR_LOG}')
    print(fails[['subject', 'task', 'run', 'error']].head(20).to_string(index=False))
else:
    print('🎉 No failures.')

In [ ]:
ok = log_df[log_df['status'] == 'ok']
if len(ok):
    print(f'Across {len(ok)} successful files:\n')
    print('Epochs / amplitude:')
    print(ok[['n_epochs', 'mean_p2p_uv', 'max_p2p_uv', 'total_duration_s']].describe(), '\n')
    print('Total epochs per task:')
    print(ok.groupby('task')['n_epochs'].sum().to_string())

In [ ]:
if len(ok):
    no_epo = ok[ok['n_epochs'] == 0]
    if len(no_epo):
        print(f'⚠️  {len(no_epo)} files produced 0 epochs:')
        print(no_epo[['subject', 'task', 'run', 'epoch_mode', 'warning']]
              .head(20).to_string(index=False))
    else:
        print('✅ Every successful file produced at least one epoch.')